### import packages and grab data

In [ ]:
import sys
sys.path.extend(['/Users/amonast/Documents/GitHub/Engram_2P/Engram_2P'])
from utilities.traces import mean_rates,zscore_traces
from utilities.animal import animal
from behavior.running import get_rest_array
from scipy.stats import zscore
import pandas as pd
import numpy as np
import statsmodels.formula.api as smf
import statsmodels.api as sm
import warnings
warnings.simplefilter(action="ignore", category=FutureWarning)
import ast
import pingouin as pg
import seaborn as sb
from matplotlib import rcParams
import matplotlib.pyplot as plt
#%%
data_dir='/Users/amonast/Desktop/Tone2P'
file_key='/Users/amonast/Desktop/Tone2P/Data_info_TFC.csv'
sessions=['Recall1']
animals = ['997B','639N', 'M2L','M1N','F5L','F7N','M8BL2','194L','M5L','M9BR2','939L']

ensemble_colors = {'Tag'     : '#F37243',
                    'Non-tag': '#00ABC8',
                    'Whole Population':'black'}
gray='#424949'
colors = ['#00ABC8','#F37243'] #Non-engram, Engram
palette={False:colors[0],True:colors[1]}
plt.style.use('/Users/amonast/Documents/GitHub/CA1_Engram_Dynamics/figures/paper_style.mplstyle')
gray='#424949'

#  Whole session

In [ ]:
#%% total session
df_tot = pd.DataFrame()
for ani in animals:
    mouse = animal(ani,'FOV1',file_key,data_dir)
    traces_df = mouse.load_traces(sessions=sessions,split=False,signal='events')
    on_times,off_times = mouse.load_tone_times()
    zscored_traces = zscore_traces(traces_df)
    rates_tot = mean_rates(traces_df,session_name=sessions[0],time_ranges=None,weighted=True)
    rates_tot['animal']=[ani]*len(rates_tot)
    df_tot = pd.concat([df_tot,rates_tot],ignore_index=True)


In [ ]:
df_tot.to_csv(f"{data_dir}/Analysis/rates/rate_df_Recall1_wholesession_period_n11.csv")

#### linear mixed effects model - whole session
##### 

In [ ]:
df_tot['animal_cell'] = df_tot['animal'] + "_" + df_tot['cell']
df_tot['log_rate'] = np.log(df_tot['mean_rate'])
df_tot = df_tot[np.isfinite(df_tot['log_rate'])]

model = sm.MixedLM.from_formula(
    "log_rate ~ C(is_engram)",
    data=df_tot,
    groups=df_tot["animal"],  # Cell is nested within animal
    re_formula="~1").fit()
model.summary()

In [ ]:
fig,ax=plt.subplots(ncols=2,nrows=1,figsize=(5,3))
sb.violinplot(data=df_tot,ax=ax[0],y='log_rate',hue='is_engram',palette=palette,inner='box',legend=False,
              gap=0.1,density_norm='width',split=True,alpha=0.8,
              inner_kws=dict(box_width=7, whis_width=2, color="0.1",alpha=0.9))
#ax[0].set_ylim(-.1,1)
sb.pointplot(data=df_tot,y='log_rate',hue='is_engram',palette=palette,ax=ax[1],dodge=.5,
            errorbar='ci',markersize=8,capsize=.1,legend=False,linewidth=2)
sb.despine()
for a in ax.flatten():
    a.set_ylabel('log(Event Rate)')
plt.tight_layout()

In [ ]:
# #####  Tone Period (non-zscored)
tone_period = pd.DataFrame()
for ani in animals:
    mouse = animal(ani, 'FOV1', file_key, data_dir)
    traces_df = mouse.load_traces(sessions=sessions, split=False, signal='events')
    on_times, off_times = mouse.load_tone_times()
    rates_tone = mean_rates(traces_df, session_name='Recall1',
                            time_ranges=[(on_times[0], off_times[-1] + 60)],
                            weighted=True)
    rates_tone['animal'] = [ani] * len(rates_tone)
    tone_period = pd.concat([tone_period, rates_tone], ignore_index=True)

# ##### Tone Period (z-scored)
tone_period_z = pd.DataFrame()
for ani in animals:
    mouse = animal(ani, 'FOV1', file_key, data_dir)
    traces_df = mouse.load_traces(sessions=sessions, split=False, signal='events')
    zscored_traces = zscore_traces(traces_df)
    on_times, off_times = mouse.load_tone_times()
    z_tones = mean_rates(zscored_traces, session_name='Recall1',
                         time_ranges=[(on_times[0], off_times[-1] + 60)],
                         weighted=True)
    z_tones['animal'] = [ani] * len(z_tones)
    tone_period_z = pd.concat([tone_period_z, z_tones], ignore_index=True)

# ##### Baseline (non-zscored)
baseline = pd.DataFrame()
for ani in animals:
    mouse = animal(ani, 'FOV1', file_key, data_dir)
    on_times, off_times = mouse.load_tone_times()
    traces_df = mouse.load_traces(sessions=sessions, split=False, signal='events')
    rate_df = mean_rates(traces_df, session_name='Recall1',
                         time_ranges=[(0, on_times[0] - 60)],
                         weighted=True)
    rate_df['animal'] = [ani] * len(rate_df)
    baseline = pd.concat([baseline, rate_df], ignore_index=True)

# ##### Baseline (z-scored,activity)
baseline_z = pd.DataFrame()
for ani in animals:
    mouse = animal(ani, 'FOV1', file_key, data_dir)
    on_times, off_times = mouse.load_tone_times()
    traces_df = mouse.load_traces(sessions=sessions, split=False, signal='events')
    zscored_traces = zscore_traces(traces_df)
    z_baseline = mean_rates(zscored_traces, session_name='Recall1',
                            time_ranges=[(0, on_times[0] - 60)],
                            weighted=True)
    z_baseline['animal'] = [ani] * len(z_baseline)
    baseline_z = pd.concat([baseline_z, z_baseline], ignore_index=True)

baseline['epoch']='baseline'
tone_period['epoch']='tone_total'
baseline_z['epoch']='baseline'
tone_period_z['epoch']='tone_total'
df = pd.concat([baseline,tone_period])
z_df = pd.concat([baseline_z,tone_period_z])
#
df["animal_cell"] = df["animal"] + "_" + df["cell"]
baseline['animal_cell'] = baseline['animal']+'_'+baseline['cell']

df.to_csv(f"{data_dir}/Analysis/rates/baseline_tones_split_updated.csv")

In [ ]:
model = smf.mixedlm(
    "mean_rate ~ is_engram * epoch",
    data=df,
    groups=df["animal"],).fit()
model.summary()

# Baseline rates 

In [ ]:
df_baseline = df.loc[df['epoch']=='baseline']
df_baseline['log_rate']=np.log(df_baseline['mean_rate'])
df_baseline=df_baseline[np.isfinite(df_baseline['log_rate'])]

fig,ax=plt.subplots(ncols=2,nrows=1,figsize=(5,3))
sb.violinplot(data=df_baseline,y='log_rate',hue='is_engram',palette=palette,inner='box',
              alpha=.7,split=True,density_norm='area',bw_adjust=1,ax=ax[0],gap=.1)
# sb.pointplot(data=df_baseline,y='log_rate',hue='is_engram',palette=palette,ax=ax[0],
#             markersize=1,markers='o',legend=False,dodge=0.1,errorbar='ci',capsize=.01,err_kws={'linewidth':1})
sb.pointplot(data=df_baseline,y='log_rate',hue='is_engram',palette=palette,ax=ax[1],
                markers='o',legend=False,dodge=.5,
                errorbar='ci',markersize=8,capsize=.1,linewidth=2)
sb.despine()
plt.tight_layout()
plt.savefig("/Users/amonast/Documents/GitHub/CA1_Engram_Dynamics/figures/SuppFig6B.svg",transparent=True)

In [ ]:
model = smf.mixedlm(
    "log_rate ~ is_engram",
    data=df_baseline,
    groups=df_baseline["animal"],  # Unique identifier per cell
    re_formula="~1").fit()
model.summary()

# Across Trials (trial dataframe)

In [ ]:
#### cells on all trials ##### 
trial_df = pd.DataFrame()
for ani in animals:
    ani_df = pd.DataFrame()
    mouse = animal(ani,'FOV1',file_key,data_dir)
    on_times,off_times = mouse.load_tone_times()
    traces_df=mouse.load_traces(sessions=sessions,split=False,signal='events')
    for i in range(len(on_times)):

        rate_df = mean_rates(traces_df,session_name='Recall1',
                             time_ranges=[(on_times[i],off_times[i])],
                             weighted=True)
        rate_df['animal']=ani
        rate_df['trial']=i+1
        #rate_df['mean_rate(z-score)']=zscore(rate_df['mean_rate'])
        ani_df=pd.concat([ani_df,rate_df])
    trial_df=pd.concat([trial_df,ani_df],ignore_index=True)

trial_df.to_csv(f"{data_dir}/Analysis/rates/rate_trials_{sessions[0]}_cells.csv")
trial_df = trial_df[trial_df["mean_rate"] != 0] #drop 0 rate cells.

    #### zscored rates (rates z-scored across all cells within each animal)
zscored = []
for ani in trial_df.animal.unique():
    df_a = trial_df.loc[trial_df['animal']==ani]
    z = zscore(df_a['mean_rate'])
    zscored.extend(z)
trial_df['zscore_rate']=zscored

    ##### cells averaged across trials #### 
cell_averages = trial_df.groupby([ 'animal','is_engram','cell',])['mean_rate'].mean().reset_index()
cell_averages['epoch']='trial_avg'
cell_averages['animal_cell'] = cell_averages['animal']+'_'+cell_averages['cell']
                
                #### animal, cells across trials  ####
ani_trials= trial_df.groupby(['animal','trial','is_engram']).mean(['mean_rate','zscore_rate']).reset_index()
ani_trials.to_csv(f"{data_dir}/Analysis/rates/rate_trials_{sessions[0]}_animals.csv")
#animal mean across all trials 
mean_across_trials = ani_trials.groupby(['animal','is_engram']).mean(['mean_rate','zscore_rate']).reset_index()

                ### mean across all cells first 3 trials ###
first_3trials = ani_trials.loc[ani_trials['trial']<=3]
mean_across_3trials = first_3trials.groupby(['animal','is_engram']).mean().reset_index()
mean_across_3trials.to_csv(f"{data_dir}/Analysis/rates/rate_first3trials_{sessions[0]}_animals")

In [ ]:
baseline['log_rate'] = np.log(baseline['mean_rate'])
cell_averages['log_rate'] = np.log(cell_averages['mean_rate'])

# Baseline vs Trial-Avg Rates

In [ ]:
mean_baseline = baseline.groupby(['animal','is_engram']).mean(['mean_rate','zscore_rate']).reset_index()
mean_baseline['is_engram'] = pd.Categorical(mean_baseline['is_engram'], ordered=True)

df1 = mean_baseline.copy()
df1['epoch']='baseline'
df2 = mean_across_trials.copy().drop(['trial'],axis=1)
df2['epoch']='trial_avg'
baseline_v_trialavg = pd.concat([df1,df2],ignore_index=True)
baseline_v_trialavg.to_csv(f"{data_dir}/Analysis/rates/recall1_baselinevtrialavg.csv")

In [ ]:
plt.figure(figsize=(5,5))
rcParams['axes.linewidth'] = 1.5 # set the value globally

#sb.barplot(data=baseline_v_trialavg,x='epoch',y='mean_rate',hue='is_engram',errorbar='se',palette=colors,capsize=.1,width=.7,edgecolor='k',linewidth=.5,err_kws={'linewidth':1,'color':'k'})
sb.boxplot(data=baseline_v_trialavg,x='epoch',y='mean_rate',hue='is_engram',palette=colors,width=.7,linewidth=1.5,showfliers=False)
sb.stripplot(data=baseline_v_trialavg,x='epoch',y='mean_rate',hue='is_engram',palette=colors,edgecolor='k',linewidth=1,dodge=.2,jitter=0,alpha=.7,size=5,legend=False)


# Add lines connecting "engram" and "non-engram" for each animal
for i, animal in enumerate(baseline_v_trialavg['animal'].unique()):
    animal_data = baseline_v_trialavg[baseline_v_trialavg['animal'] == animal]
    for epoch in animal_data['epoch'].unique():
        if epoch=='baseline':
            x1,x2=-.2,.2
        else:
            x1,x2=.8,1.2
        engram_data = animal_data[(animal_data['epoch'] == epoch) & (animal_data['is_engram'] == 1)]
        nonengram_data = animal_data[(animal_data['epoch'] == epoch) & (animal_data['is_engram'] == 0)]
        
        if not engram_data.empty and not nonengram_data.empty:
            plt.plot([x1, x2], [nonengram_data['mean_rate'].values[0],engram_data['mean_rate'].values[0],], 
                     color=gray, linewidth=1, alpha=0.7)
plt.legend(loc='upper right',bbox_to_anchor=(1.2,1),frameon=False)
plt.xticks(ticks=[0,1],labels=['Baseline','CS-averaged'])
plt.ylabel('Mean Event Rate')
sb.despine()
plt.tight_layout()
plt.savefig(f"{data_dir}/Analysis/rates/ani_mean_baseline_v_trialavg.svg",transparent=True)

# Zscore to Baseline


In [ ]:
# Normalize each cell by the animal-level baseline mean and std (computed across all cells in that animal).
# This removes between-animal scale differences while preserving the engram vs. non-engram
# baseline difference and trial-to-trial variability.
#
# z = (trial_rate_cell_i - animal_baseline_mean) / animal_baseline_std

trial_df_z = pd.DataFrame()
for ani in animals:
    mouse = animal(ani, 'FOV1', file_key, data_dir)
    on_times, off_times = mouse.load_tone_times()
    traces_df = mouse.load_traces(sessions=sessions, split=False, signal='events')

    # Compute baseline rates for all cells in this animal
    baseline_rates = mean_rates(traces_df, session_name='Recall1',
                                time_ranges=[(0, on_times[0])],
                                weighted=True)

    # Single mean and std across all cells in this animal's baseline
    pop_mean = baseline_rates['mean_rate'].mean()
    pop_std  = baseline_rates['mean_rate'].std()

    # Compute trial rates and z-score using population baseline stats
    for i in range(len(on_times)):
        rate_df = mean_rates(traces_df, session_name='Recall1',
                             time_ranges=[(on_times[i], off_times[i])],
                             weighted=True)
        rate_df['animal']       = ani
        rate_df['trial']        = i + 1
        rate_df['trial_rate_z'] = (rate_df['mean_rate'] - pop_mean) / pop_std
        trial_df_z = pd.concat([trial_df_z, rate_df], ignore_index=True)

trial_df_z = trial_df_z[trial_df_z['mean_rate'] != 0].reset_index(drop=True)

# Aggregate to animal level for plotting
ani_trials_z = trial_df_z.groupby(['animal', 'trial', 'is_engram']).mean(numeric_only=True).reset_index()


In [ ]:
ani_trials_z.to_csv(f"{data_dir}/Analysis/rates/rate_trials_{sessions[0]}_animals_norm2baseline.csv")

In [ ]:

hue_order = [False,True]
palette=['#00ABC8','#F37243']

##################### Mean across trials (dropped inactive cells)
fig,ax=plt.subplots(ncols=1,nrows=1,figsize=(4,2.8))
sb.pointplot(data=ani_trials_z,y='trial_rate_z',x='trial',hue='is_engram',errorbar='se',
                capsize=0.1,
                markersize=6,
                palette=palette,
                markers='o',
                linewidth=2,
                dodge=0,
                legend=True,
                err_kws=dict(lw=1.))
plt.legend(frameon=False,bbox_to_anchor=(.9,1))
plt.xlabel('Trial')
plt.ylabel('Event Rate (z-score)')
sb.despine()
plt.tight_layout()
plt.savefig("/Users/amonast/BOSTON UNIVERSITY Dropbox/Amy Monasterio/Manuscripts/Engram2P/Figures/RevisionFigures/Figure5_Supp6/zscore_across_trials.svg" ,transparent=True)      
#%%

In [ ]:
aov=pg.rm_anova(data=ani_trials_z,within=['trial','is_engram'],dv='trial_rate_z',subject='animal')
aov